# runctrl.vars.nml generator 
The script generates various namelists for CORDEX CMIP6 variables based on the list published on zenodo:
https://zenodo.org/records/8414798

To run the script a csv file containing data from the file downloaded from zenodo is necessary to be placed in the running directory, and the list of the variables which a user want to process with pCMORizer.exe. 
The csv file is CORDEX_CMIP6_variables.csv

Contact: milovacj@unican.es

Define all the functions:

In [26]:
def generate_namelist(varlist, nvars):
    
    # Set header and footer:
    header = "&vars\n"
    footer = "/"
    
    # Set titles:
    titles = [
        '   cordexID',
        '   var_wrf',
        '   var_cmip',
        '   standard_name',
        '   long_name',
        '   units',
        '   height',
        '   plevel',
        '   positive',
        '   time1hr',
        '   cm1hr',
        '   time3hr',
        '   cm3hr',
        '   time6hr',
        '   cm6hr',
        '   timeDay',
        '   cmDay',
        '   timeMon',
        '   cmMon',
        '   timeSea',
        '   cmSea',
        '   filetype',
        '   interpolate'
    ]
    
    # Calculate the maximum length of each column
    column_length = max(len(title) for title in titles) + 5  # Add padding for better readability
    
    # Create lines in the template
    variable_lines = ""
    for title,j in zip(titles,range(0,len(titles))):
        variable_lines += "{:<{}} = ".format(title,20)
        for i in range(0,nvars):
            if i > len(varlist)-1:
                nspace = max(len(varline) for varline in fill_varlist) + 1  # Add padding for better readability
                variable_lines += "{:<{}},".format(fill_varlist[j], nspace)
            else:
                nspace = max(len(varline) for varline in varlist[i]) + 1  # Add padding for better readability
                variable_lines += "{:<{}},".format(varlist[i][j], nspace)
            
        variable_lines += "\n"

    # Create the template
    template = f"{header}{variable_lines}{footer}"
    return template

def map_to_tf(value):
    return {'x': 'T', 'other': 'F'}.get(value, 'F')

def find_height(name):
    numbers = ''.join(char for char in name if char.isdigit())
    value = numbers if numbers else "-999"
    if value == "999":
        value = "-999"
    return value

def create_vararray(varname, filepath):
    import csv

    def read_variable_info(filepath):
        data = []

        with open(filepath, 'r') as file:
            reader = csv.DictReader(file, delimiter=',')
            for row in reader:
                data.append(row)

        return data

    filepath = filepath  # Change this to the actual file path
    variable_info = read_variable_info(filepath)

    # Print the variable information
    for entry in variable_info:
        if entry['output variable name']==varname:
            if entry['ag']=="a":
                cm1hr = cm3hr = cm6hr = cmDay = "'mean'"
            elif varname=="sund":
                cm1hr = cm3hr = cm6hr = cmDay = "'sum'"
            elif "max" in varname:
                cm1hr = cm3hr = cm6hr = cmDay = "'max'"
            elif "min" in varname:
                cm1hr = cm3hr = cm6hr = cmDay = "'min'"
            else:
                cm1hr = cm3hr = cm6hr = cmDay = "'point'"
                
            if "down" in entry['standard_name'] or \
            "incoming" in entry['standard_name'] or \
            varname=="rsdt":
                positive = f"'down'"
            elif "upward" in entry['standard_name'] or \
            "upwelling" in entry['standard_name'] or \
            "outgoing"  in entry['standard_name']:
                positive = f"'up'"
            else:
                positive = f"'-999'"  
                
            height = find_height(entry['WRF variable']) 
            plevel = "-999"
            
            for name in ["ua","va","wa","ta","zg"]:
                if name in varname:
                    if "m" in varname:
                        height = find_height(varname)
                        plevel = "-999"
                    else:
                        height = "-999"
                        plevel = find_height(varname)
            
            vararray = ["999",\
                        f"'{entry['WRF variable']}'",\
                        f"'{entry['output variable name']}'",\
                        f"'{entry['standard_name']}'",\
                        f"'{entry['long_name']}'",\
                        f"'{entry['units']}'",\
                       height,\
                       plevel,\
                       positive,\
                       map_to_tf(entry['1hr']),\
                       cm1hr,\
                       map_to_tf(entry['3hr']),\
                       cm3hr,\
                       map_to_tf(entry['6hr']),\
                       cm6hr,\
                       map_to_tf(entry['day']),\
                       cmDay,\
                       map_to_tf(entry['mon']),\
                       "'mean'",\
                       "F",\
                       "'mean'",\
                       "'s'",\
                       "T",\
                      ]   
            return vararray

def create_multi_vararray(filepath,*varnames):
    result = []
    
    for varname in varnames:
        vararray = create_vararray(varname,filepath)
        result.append(vararray)
    
    return result

Complete lists of core, trier1, and trier2 CORDEX-CMIP6 variables:

In [27]:
varnames_core = ('tas', 'tasmax', 'tasmin', 'pr', 'evspsbl', 'huss', 'hurs', 'ps', 
                 'psl', 'sfcWind', 'uas', 'vas', 'clt', 'rsds', 'rlds', 'orog', 'sftlf')

varnames_trier1 = ('ts', 'tsl', 'prc', 'prhmax', 'prsn', 'mrros', 'mrro', 'snm', 'tauu', \
                   'tauv', 'sfcWindmax', 'sund', 'rsdsdir', 'rsus', 'rlus', 'rlut', 'rsdt', \
                   'rsut', 'hfls', 'hfss', 'mrfso', 'mrfsos', 'mrsfl', 'mrso', 'mrsos', \
                   'mrsol', 'snw', 'snc', 'snd', 'siconca', 'zmla', 'prw', 'clwvi', 'clivi', \
                   'ua1000', 'ua925', 'ua850', 'ua700', 'ua600', 'ua500', 'ua400', 'ua300', \
                   'ua250', 'ua200', 'va1000', 'va925', 'va850', 'va700', 'va600', 'va500', \
                   'va400', 'va300', 'va250', 'va200', 'ta1000', 'ta925', 'ta850', 'ta700', \
                   'ta600', 'ta500', 'ta400', 'ta300', 'ta250', 'ta200', 'hus1000', 'hus925', \
                   'hus850', 'hus700', 'hus600', 'hus500', 'hus400', 'hus300', 'hus250', \
                   'hus200', 'zg1000', 'zg925', 'zg850', 'zg700', 'zg600', 'zg500', 'zg400', \
                   'zg300', 'zg250', 'zg200', 'wa1000', 'wa925', 'wa850', 'wa700', 'wa600', \
                   'wa500', 'wa400', 'wa300', 'wa250', 'wa200', 'ua50m', 'ua100m', 'ua150m', \
                   'va50m', 'va100m', 'va150m', 'ta50m', 'hus50m')

varnames_trier2 = ('evspsblpot', 'wsgsmax', 'clh', 'clm', 'cll', 'rsdscs', 'rldscs', \
                   'rsuscs', 'rluscs', 'rsutcs', 'rlutcs', 'z0', 'CAPE', 'LI', 'CIN', \
                   'CAPEmax', 'LImax', 'CINmax', 'od550aer', 'ua150', 'ua100', 'ua70', \
                   'ua50', 'ua30', 'ua20', 'ua10', 'va150', 'va100', 'va70', 'va50', \
                   'va30', 'va20', 'va10', 'ta150', 'ta100', 'ta70', 'ta50', 'ta30', \
                   'ta20', 'ta10', 'hus150', 'hus100', 'hus70', 'hus50', 'hus30', \
                   'hus20', 'hus10', 'zg150', 'zg100', 'zg70', 'zg50', 'zg30', 'zg20', \
                   'zg10', 'wa150', 'wa100', 'wa70', 'wa50', 'wa30', 'wa20', 'wa10', \
                   'ua750', 'va750', 'ta750', 'hus750', 'zg750', 'wa750', 'ua200m', \
                   'ua250m', 'ua300m', 'va200m', 'va250m', 'va300m', 'sftgif', 'mrsofc', \
                   'rootd', 'sftlaf', 'sfturf', 'dtb', 'areacella')

# Variables separated in smaller chunks for trier1 and trier2
varnames_trier1_sfc = ('ts', 'tsl', 'prc', 'prhmax', 'prsn', 'mrros', 'mrro', 'snm', 'tauu', \
                   'tauv', 'sfcWindmax', 'sund', 'rsdsdir', 'rsus', 'rlus', 'rlut', 'rsdt', \
                   'rsut', 'hfls', 'hfss', 'mrfso', 'mrfsos', 'mrsfl', 'mrso', 'mrsos', \
                   'mrsol', 'snw', 'snc', 'snd', 'siconca', 'zmla', 'prw', 'clwvi', 'clivi')

varnames_trier1_int = ('prw', 'clwvi', 'clivi')

varnames_trier1_plevel = ( 'ua1000', 'ua925', 'ua850', 'ua700', 'ua600', 'ua500', 'ua400', 'ua300', \
                   'ua250', 'ua200', 'va1000', 'va925', 'va850', 'va700', 'va600', 'va500', \
                   'va400', 'va300', 'va250', 'va200', 'ta1000', 'ta925', 'ta850', 'ta700', \
                   'ta600', 'ta500', 'ta400', 'ta300', 'ta250', 'ta200', 'hus1000', 'hus925', \
                   'hus850', 'hus700', 'hus600', 'hus500', 'hus400', 'hus300', 'hus250', \
                   'hus200', 'zg1000', 'zg925', 'zg850', 'zg700', 'zg600', 'zg500', 'zg400', \
                   'zg300', 'zg250', 'zg200', 'wa1000', 'wa925', 'wa850', 'wa700', 'wa600', \
                   'wa500', 'wa400', 'wa300', 'wa250', 'wa200', 'ua50m', 'ua100m', 'ua150m', \
                   'va50m', 'va100m', 'va150m', 'ta50m', 'hus50m')

varnames_trier1_height = ('ua50m', 'ua100m', 'ua150m', 'va50m', 'va100m', 'va150m', 'ta50m', 'hus50m')

varnames_trier2_sfc = ('evspsblpot', 'wsgsmax', 'rsdscs', 'rldscs', 'rsuscs', 'rluscs', 'rsutcs', 'rlutcs', 'z0') 

varnames_trier2_int = ( 'clh', 'clm', 'cll', 'CAPE', 'LI', 'CIN','CAPEmax', 'LImax', 'CINmax', 'od550aer')
                       
varnames_trier2_plevel = ( 'ua150', 'ua100', 'ua70', \
                   'ua50', 'ua30', 'ua20', 'ua10', 'va150', 'va100', 'va70', 'va50', \
                   'va30', 'va20', 'va10', 'ta150', 'ta100', 'ta70', 'ta50', 'ta30', \
                   'ta20', 'ta10', 'hus150', 'hus100', 'hus70', 'hus50', 'hus30', \
                   'hus20', 'hus10', 'zg150', 'zg100', 'zg70', 'zg50', 'zg30', 'zg20', \
                   'zg10', 'wa150', 'wa100', 'wa70', 'wa50', 'wa30', 'wa20', 'wa10', \
                   'ua750', 'va750', 'ta750', 'hus750', 'zg750', 'wa750') 
                        
varnames_trier2_height = ('ua200m', 'ua250m', 'ua300m', 'va200m', 'va250m', 'va300m')
                          
varnames_trier2_fx = ( 'sftgif', 'mrsofc', 'rootd', 'sftlaf', 'sfturf', 'dtb', 'areacella')

Running the script:

In [28]:
# csv files with the CORDEX variables
filepath   = "CORDEX_CMIP6_variables.csv"

# E.g. generate namelist for core variables
varnames=varnames_core
fname = 'core'

varlist_array = create_multi_vararray(filepath, *varnames)
namelist_content = generate_namelist(varlist_array, len(varlist_array))

# Specify the file path
file_path = f'runctrl.vars.{fname}.nml'

# Write the content to the file
with open(file_path, "w") as file:
    file.write(namelist_content)

print(f"File '{file_path}' created successfully.")

File 'runctrl.vars.core.nml' created successfully.
